### Model Subclassing & Custom Training Loop Using Reuters Dataset

In [2]:
#importing libraries

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import time

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Softmax

In [11]:
#defining the custom layers and model

class MyLayer(Layer):

    def __init__(self, units):
        super(MyLayer, self).__init__()
        self.units = units

    def build(self, input_shape):

        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                    initializer="random_normal",
                    name="kernel")
        self.b = self.add_weight(shape=(self.units, ),
                    initializer="zeros",
                    name="bias")

    def call(self, inputs):
        return tf.matmul(inputs, self.w) + self.b


class MyDropout(Layer):

    def __init__(self, rate):
        super(MyDropout, self).__init__()
        self.rate = rate

    def call(self, inputs):
        return tf.nn.dropout(inputs, rate=self.rate)  


class MyModel(Model):

    def __init__(self, units_1, units_2, units_3):
        super(MyModel, self).__init__()
        self.layer_1 = MyLayer(units_1)  
        self.layer_2 = MyLayer(units_2)
        self.layer_3 = MyLayer(units_3)
        self.dropout_1 = MyDropout(0.5)
        self.dropout_2 = MyDropout(0.5)
        self.softmax = Softmax()

    def call(self, inputs):
        #define forward pass
        x = self.layer_1(inputs)
        x = tf.nn.relu(x)
        x = self.dropout_1(x)
        x = self.layer_2(x)
        x = tf.nn.relu(x)
        x = self.dropout_2(x)
        x = self.layer_3(x)
        x = tf.nn.relu(x)
        return self.softmax(x)


In [13]:
#instantiating the model object

model = MyModel(64, 64, 46)
print(model(tf.ones((1, 10000))))
model.summary()

tf.Tensor(
[[0.02898734 0.00763412 0.02961294 0.00763412 0.06038728 0.00963221
  0.09489224 0.25611803 0.00763412 0.00763412 0.00763412 0.01437753
  0.00837879 0.00763412 0.00763412 0.00763412 0.00763412 0.00763412
  0.00763412 0.01125371 0.00763412 0.00763412 0.01365206 0.00763412
  0.00763412 0.00763412 0.00763412 0.00763412 0.00870814 0.01656758
  0.03927936 0.00763412 0.00763412 0.01740373 0.01579197 0.00763412
  0.01134052 0.00763412 0.00763412 0.08378071 0.03143896 0.00880012
  0.00763412 0.04110966 0.00763412 0.00763412]], shape=(1, 46), dtype=float32)


Model: "my_model_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ my_layer_12 (MyLayer)           │ ?                      │       640,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_13 (MyLayer)           │ ?                      │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_layer_14 (MyLayer)           │ ?                      │         2,990 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dropout_8 (MyDropout)        │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ my_dropout_9 (MyDropout)        │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_4 (Softmax)             │ ?                      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 647,214 (2.47 MB)

 Trainable params: 647,214 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
#loading the reuters dataset

from tensorflow.keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

class_names = ['cocoa','grain','veg-oil','earn','acq','wheat','copper','housing','money-supply',
   'coffee','sugar','trade','reserves','ship','cotton','carcass','crude','nat-gas',
   'cpi','money-fx','interest','gnp','meal-feed','alum','oilseed','gold','tin',
   'strategic-metal','livestock','retail','ipi','iron-steel','rubber','heat','jobs',
   'lei','bop','zinc','orange','pet-chem','dlr','gas','silver','wpi','hog','lead']

d:\Tensorflow_Works\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [20]:
#printing the class of first sample

print("Label: {}".format(class_names[train_labels[0]]))

Label: earn


In [37]:
#loading the reuters word index

word_to_index = reuters.get_word_index()
invert_word_index = dict([(value, key) for (key, value) in word_to_index.items()])
text_news = " ".join([invert_word_index.get(value-3, "?") for value in train_data[0]])
text_news 

'? ? ? said as a result of its december acquisition of space co it expects earnings per share in 1987 of 1 15 to 1 30 dlrs per share up from 70 cts in 1986 the company said pretax net should rise to nine to 10 mln dlrs from six mln dlrs in 1986 and rental operation revenues to 19 to 22 mln dlrs from 12 5 mln dlrs it said cash flow per share this year should be 2 50 to three dlrs reuter 3'

In [38]:
#defining a function that encodes the data into a "bag of words" representation

def bag_of_words(text_samples, elements=10000):
    output = np.zeros((len(text_samples), elements))
    for i, word in enumerate(text_samples):
        output[i, word] =1.
    return output

x_train = bag_of_words(train_data)
x_test = bag_of_words(test_data)

print("Shape of x_train:", x_train.shape)
print("Shape of x_test:", x_test.shape)


Shape of x_train: (8982, 10000)
Shape of x_test: (2246, 10000)


In [39]:
x_train[0]

array([0., 1., 1., ..., 0., 0., 0.], shape=(10000,))

In [40]:
#define the loss function and optimizer

loss_object = tf.keras.losses.SparseCategoricalCrossentropy()

def loss(model, x, y, wd):
    kernel_variables=[]
    for l in model.layers:
        for w in l.weights:
            if "kernel" in w.name:
                kernel_variables.append(w)
    wd_penalty = wd * tf.reduce_sum([tf.reduce_sum(tf.square(k) for k in kernel_variables)])
    y_ = model(x)
    return loss_object(y_true=y, y_pred=y_) + wd_penalty 

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)               